In [7]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error

# Load EDA dataset
path = Path("data/eda.parquet")
if not path.exists():
    path = Path("notebooks/data/eda.parquet")

df = pd.read_parquet(path).sort_index()

# Recreate existing features
df["lag_1"] = df["Appliances"].shift(1)
df["lag_6"] = df["Appliances"].shift(6)
df["lag_144"] = df["Appliances"].shift(144)

df["rolling_mean_6"] = df["Appliances"].shift(1).rolling(6).mean()
df["rolling_std_6"] = df["Appliances"].shift(1).rolling(6).std()
df["rolling_mean_144"] = df["Appliances"].shift(1).rolling(144).mean()
df["rolling_std_144"] = df["Appliances"].shift(1).rolling(144).std()

df["hour"] = df.index.hour
df["day_of_week"] = df.index.dayofweek
df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

df["diff_1"] = df["Appliances"].shift(1).diff(1)
df["diff_6"] = df["Appliances"].shift(1).diff(6)
df["diff_144"] = df["Appliances"].shift(1).diff(144)

df = df.dropna()

# Chronological split
train_df, remaining_df = train_test_split(df, shuffle=False, test_size=0.3)
val_df, test_df = train_test_split(remaining_df, shuffle=False, test_size=0.5)

# Original feature set
features = train_df.columns.drop("Appliances").tolist()

# Candidate interactions
interactions = {
    "Outdoor Temp x Humidity": ("T_out", "RH_out", "multiply"),
    "Room 1 Temp x Humidity": ("T1", "RH_1", "multiply"),
    "Room 2 Temp x Humidity": ("T2", "RH_2", "multiply"),
    "Room 3 Temp x Humidity": ("T3", "RH_3", "multiply"),
    "Room 4 Temp x Humidity": ("T4", "RH_4", "multiply"),
    "Room 5 Temp x Humidity": ("T5", "RH_5", "multiply"),
    "Room 6 Temp x Humidity": ("T6", "RH_6", "multiply"),
    "Room 7 Temp x Humidity": ("T7", "RH_7", "multiply"),
    "Room 8 Temp x Humidity": ("T8", "RH_8", "multiply"),
    "Room 9 Temp x Humidity": ("T9", "RH_9", "multiply"),

    "Indoor Outdoor Temp Difference": ("T1", "T_out", "subtract"),
    "Outdoor Temp x Wind": ("T_out", "Windspeed", "multiply")
}

def evaluate(train, val, columns):
    model = LinearRegression()
    model.fit(train[columns], train["Appliances"])
    predictions = model.predict(val[columns])
    return mean_squared_error(val["Appliances"], predictions)

# Baseline
baseline_mse = evaluate(train_df, val_df, features)
results = []

# Test one interaction at a time
for name, (col1, col2, operation) in interactions.items():
    train = train_df.copy()
    val = val_df.copy()

    if operation == "multiply":
        train["interaction"] = train[col1] * train[col2]
        val["interaction"] = val[col1] * val[col2]
    elif operation == "subtract":
        train["interaction"] = train[col1] - train[col2]
        val["interaction"] = val[col1] - val[col2]

    elif operation == "mean":
        train["interaction"] = train[col1] train[col2]
        val["interaction"] = val[col1] - val[col2]

    
    mse = evaluate(train, val, features + ["interaction"])

    results.append({
        "Interaction": name,
        "Baseline MSE": baseline_mse,
        "New MSE": mse,
        "Improvement": baseline_mse - mse
    })

results_df = pd.DataFrame(results).sort_values("New MSE")
print(results_df.round(4).to_string(index=False))



                   Interaction  Baseline MSE   New MSE  Improvement
        Room 2 Temp x Humidity     3569.0779 3525.5546      43.5233
        Room 9 Temp x Humidity     3569.0779 3564.8125       4.2655
        Room 6 Temp x Humidity     3569.0779 3566.6007       2.4772
        Room 1 Temp x Humidity     3569.0779 3567.4716       1.6063
        Room 4 Temp x Humidity     3569.0779 3567.8528       1.2251
       Outdoor Temp x Humidity     3569.0779 3568.6963       0.3816
Indoor Outdoor Temp Difference     3569.0779 3569.0779      -0.0000
           Outdoor Temp x Wind     3569.0779 3570.3815      -1.3036
        Room 8 Temp x Humidity     3569.0779 3571.4136      -2.3357
        Room 5 Temp x Humidity     3569.0779 3572.2531      -3.1752
        Room 7 Temp x Humidity     3569.0779 3572.6318      -3.5539
        Room 3 Temp x Humidity     3569.0779 3578.5284      -9.4505


In [8]:

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Load dataset
path = Path("data/eda.parquet")
if not path.exists():
    path = Path("notebooks/data/eda.parquet")

df = pd.read_parquet(path).sort_index()

# Existing feature engineering
for lag in [1, 6, 144]:
    df[f"lag_{lag}"] = df["Appliances"].shift(lag)

for window in [6, 144]:
    df[f"rolling_mean_{window}"] = df["Appliances"].shift(1).rolling(window).mean()
    df[f"rolling_std_{window}"] = df["Appliances"].shift(1).rolling(window).std()

df["hour"] = df.index.hour
df["day_of_week"] = df.index.dayofweek
df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

for lag in [1, 6, 144]:
    df[f"diff_{lag}"] = df["Appliances"].shift(1).diff(lag)

# Store original feature names
base_features = df.columns.drop("Appliances").tolist()

# Groups of related sensors
temp_cols = [f"T{i}" for i in range(1, 10)]
humidity_cols = [f"RH_{i}" for i in range(1, 10)]

# Mean features
df["temp_mean"] = df[temp_cols].mean(axis=1)
df["humidity_mean"] = df[humidity_cols].mean(axis=1)

# Standard deviation features
df["temp_std"] = df[temp_cols].std(axis=1)
df["humidity_std"] = df[humidity_cols].std(axis=1)

# Range features
df["temp_range"] = df[temp_cols].max(axis=1) - df[temp_cols].min(axis=1)
df["humidity_range"] = df[humidity_cols].max(axis=1) - df[humidity_cols].min(axis=1)

# Minimum and maximum features
df["temp_min"] = df[temp_cols].min(axis=1)
df["temp_max"] = df[temp_cols].max(axis=1)
df["humidity_min"] = df[humidity_cols].min(axis=1)
df["humidity_max"] = df[humidity_cols].max(axis=1)

# Temperature and humidity interactions for all rooms
for i in range(1, 10):
    df[f"T{i}_RH{i}"] = df[f"T{i}"] * df[f"RH_{i}"]

# Aggregated interactions
df["mean_temp_x_humidity"] = df["temp_mean"] * df["humidity_mean"]
df["temp_std_x_humidity_std"] = df["temp_std"] * df["humidity_std"]
df["temp_mean_x_outdoor_humidity"] = df["temp_mean"] * df["RH_out"]
df["outdoor_temp_x_humidity"] = df["T_out"] * df["RH_out"]
df["outdoor_temp_x_wind"] = df["T_out"] * df["Windspeed"]

# Nonlinear temperature and humidity differences
df["temp_gap_squared"] = (df["temp_mean"] - df["T_out"]) ** 2
df["humidity_gap_squared"] = (df["humidity_mean"] - df["RH_out"]) ** 2

# Historical rolling averages for selected sensor features
for column in ["T2", "RH_2", "temp_mean", "humidity_mean"]:
    for window in [6, 144]:
        df[f"{column}_rolling_mean_{window}"] = df[column].shift(1).rolling(window).mean()

# Cyclical time features
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

# Identify new candidate features
candidate_features = [col for col in df.columns if col not in base_features + ["Appliances"]]

df = df.dropna()

# Chronological split
train_df, remaining_df = train_test_split(df, shuffle=False, test_size=0.3)
val_df, test_df = train_test_split(remaining_df, shuffle=False, test_size=0.5)

def evaluate(features):
    model = LinearRegression()
    model.fit(train_df[features], train_df["Appliances"])
    predictions = model.predict(val_df[features])
    return mean_squared_error(val_df["Appliances"], predictions)

# Baseline evaluation
baseline_mse = evaluate(base_features)
print(f"Baseline MSE: {baseline_mse:.4f}")
print(f"Baseline RMSE: {np.sqrt(baseline_mse):.4f}")

# Experiment 1: Evaluate individual candidate features
results = []

for feature in candidate_features:
    mse = evaluate(base_features + [feature])
    results.append({
        "Feature": feature,
        "MSE": mse,
        "Improvement": baseline_mse - mse
    })

results_df = pd.DataFrame(results).sort_values("MSE")

print("\nIndividual Feature Results:")
print(results_df.round(4).to_string(index=False))

# Experiment 2: Sequential forward feature selection
selected_features = []
remaining_features = candidate_features.copy()
current_mse = baseline_mse

max_features = 8
min_improvement = baseline_mse * 0.001

selection_history = []

for step in range(max_features):
    scores = []

    for feature in remaining_features:
        columns = base_features + selected_features + [feature]
        mse = evaluate(columns)
        scores.append((feature, mse))

    best_feature, best_mse = min(scores, key=lambda x: x[1])
    improvement = current_mse - best_mse

    if improvement < min_improvement:
        print("\nStopping: No sufficiently useful feature found.")
        break

    selected_features.append(best_feature)
    remaining_features.remove(best_feature)

    selection_history.append({
        "Step": step + 1,
        "Feature Added": best_feature,
        "Validation MSE": best_mse,
        "Improvement": improvement
    })

    current_mse = best_mse

print("\nForward Selection Results:")
print(pd.DataFrame(selection_history).round(4).to_string(index=False))

# Final comparison
final_mse = evaluate(base_features + selected_features)

comparison = pd.DataFrame({
    "Model": ["Baseline", "Selected Features"],
    "MSE": [baseline_mse, final_mse],
    "RMSE": [np.sqrt(baseline_mse), np.sqrt(final_mse)]
})

print("\nFinal Comparison:")
print(comparison.round(4).to_string(index=False))

print("\nSelected Features:")
print(selected_features)


Baseline MSE: 3569.0779
Baseline RMSE: 59.7418

Individual Feature Results:
                       Feature       MSE  Improvement
                        T2_RH2 3525.5546      43.5233
                      hour_cos 3545.8029      23.2750
                      hour_sin 3559.2562       9.8217
humidity_mean_rolling_mean_144 3561.9097       7.1682
                       day_sin 3562.6466       6.4313
           RH_2_rolling_mean_6 3563.5384       5.5395
         RH_2_rolling_mean_144 3563.6811       5.3968
       temp_std_x_humidity_std 3564.0541       5.0238
                      temp_min 3564.1644       4.9136
                        T9_RH9 3564.8125       4.2655
                        T6_RH6 3566.6007       2.4772
                  humidity_max 3567.0048       2.0731
      temp_mean_rolling_mean_6 3567.1148       1.9631
                        T1_RH1 3567.4716       1.6063
                        T4_RH4 3567.8528       1.2251
       outdoor_temp_x_humidity 3568.6963       0.3816
  humi